<a href="https://colab.research.google.com/github/AhmedWaleedElFar/flyrank-ml-assignments/blob/main/work/notebooks/w03_data_contract.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/AhmedWaleedElFar/flyrank-ml-assignments/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

- One row = one pseudomnyzed content page with features aggregated from prior 90-day window (t-90 to t) and a target label predicting the decline in the next 30 days (t to t+30)

- Tables Included: fact_content_daily_performance (features), dim_content (metadata), dim_clients (history coverage).

- Tables Excluded: fact_content_query_90d (fixed 90-day window overlaps label window, which will lead to leakage).

- Decision date at t=2026-05-01, Features dates cover: 2026-02-01 to 2026-05-01 (t-90 to t), Reference window: 2026-04-01 to 2026-04-31 (t-30 to t), and target label covers: 2026-05-01 to 2026-05-31 (t to t+30).

- target Label: `decline_in_next_30d = 1 if sum(impressions)_t_to_t+30 < 0.8 * sum(impressions)_t-60_to_t-30 else 0`.

## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*


| Category | Fields |
|----------|--------|
| **Features** (8) | impressions_90d, clicks_90d, avg_position_90d, ctr_90d, content_age_days, days_since_last_update, content_type, has_ai_traffic (added for completeness only, despite being sparse)|
| **Label** | decline_in_next_30d |
| **Context** | content_hash_id, client_hash_id, report_date, gsc_data_available, ga4_data_available |
| **Excluded** | fact_query_90d columns (data leakage), word_count/char_count (missingness), search_volume/competition/cpc (missing for feedly content) |

## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

Verification checks (month=2026-03)

In [1]:
%pip -q install duckdb huggingface_hub

In [2]:
import os, getpass

# Token order: env var -> Colab Secret -> prompt (last resort).
# Use a Colab Secret named HF_TOKEN (the key panel on the left) so the prompt never
# fires: if Colab reconnects while a getpass prompt is open, the kernel waits on it
# forever ('Resuming execution...') and you have to restart the runtime.
HF_TOKEN = os.environ.get('HF_TOKEN')
if not HF_TOKEN:
    try:
        from google.colab import userdata
        HF_TOKEN = userdata.get('HF_TOKEN')
    except Exception:
        pass
HF_TOKEN = HF_TOKEN or getpass.getpass('Paste your Hugging Face READ token (hf_...): ')


In [3]:
import duckdb

con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")

REL = 'hf://datasets/FlyRank/internship-warehouse'
TABLES = {
    'dim_clients':                f"read_parquet('{REL}/dim_clients.parquet')",
    'dim_content':                f"read_parquet('{REL}/dim_content.parquet')",
    'fact_daily':                 f"read_parquet('{REL}/fact_content_daily_performance/**/*.parquet')",
    'fact_daily_sample':          f"read_parquet('{REL}/fact_content_daily_performance_sample.parquet')",
    'fact_query_90d':             f"read_parquet('{REL}/fact_content_query_90d.parquet')",
}

for name, src in TABLES.items():
    n = con.sql(f'SELECT COUNT(*) FROM {src}').fetchone()[0]
    print(f'{name:22} {n:>12,} rows')

dim_clients                     104 rows
dim_content                 519,606 rows


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

fact_daily               78,835,655 rows
fact_daily_sample        11,694,072 rows
fact_query_90d            2,414,248 rows


Query 1: Grain Check

In [7]:
query1 = con.sql(f"""
      SELECT content_hash_id, COUNT(*) c
      FROM {TABLES['fact_daily']}
      WHERE month = '2026-03'
      GROUP BY 1
      HAVING c > 1
      LIMIT 5;
""").df()

query1

,content_hash_id,c
0,content_7a105f548d9c6916,31
1,content_a3ea9792f793ec72,31
2,content_36c36abc7650d7af,31
3,content_a7da352b73b02668,31
4,content_f39be42b42a4e8f6,31


Query 2: Counts + Date Span

In [8]:
query2 = con.sql(f"""
      SELECT
          COUNT(*) AS rows,
          MIN(report_date) AS min_date,
          MAX(report_date) AS max_date,
          COUNT(DISTINCT content_hash_id) AS unique_pages,
          COUNT(DISTINCT client_hash_id) AS unique_clients
      FROM {TABLES["fact_daily"]}
      WHERE month = '2026-03';
""").df()

query2

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,rows,min_date,max_date,unique_pages,unique_clients
0,9841378,2026-03-01,2026-03-31,331437,55


Query 3: Availability (IS TRUE)

In [10]:
query3 = con.sql(f"""
SELECT
    COUNT(*) AS total_rows,
    SUM(CASE WHEN gsc_data_available IS TRUE THEN 1 ELSE 0 END) AS gsc_true,
    SUM(CASE WHEN ga4_data_available IS TRUE THEN 1 ELSE 0 END) AS ga4_true,
    SUM(CASE WHEN gsc_data_available IS TRUE AND ga4_data_available IS TRUE THEN 1 ELSE 0 END) AS both_true
FROM {TABLES['fact_daily']}
WHERE month = '2026-03';
""").df()

query3

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,total_rows,gsc_true,ga4_true,both_true
0,9841378,3611061.0,413966.0,364347.0


## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.